# 1 Initialize the Database

All the code related to data management is in the `EnvironmentData` class. This makes life easier - for example: we can send the CatsUserID once and it becomes a class property. Then, when we call other operations we don't have to send this information again.

When you create a new instance of `EnvironmentData` and there is no database, it will pull historical data and initialize the database. 

In [1]:
# Clear prior data. 
import os, sys, shutil

# Add parent directory to Python path to import EnvironmentData.
sys.path.append(os.path.dirname(os.getcwd()))

# Get the EnvironmentData class.
from EnvironmentData import EnvironmentData 

# The project adds to existing data so we need to clear that data to get a solid test from scratch.
if os.path.exists('../data'):
    shutil.rmtree('../data')
    os.makedirs('../data')

# Initialize EnvironmentData. This will run the historical data pull.
envdt = EnvironmentData(
    #days_back = 365 * 2,
    days_back = 7,
    coris_enabled = True,
    licor_enabled = True,
    conserv_enabled = True, 
    testing = True,
    #testing = False,
    # Since we are running from the experiments/ folder, we need to tell the class to use the parent directory as home.
    home_directory = ".."
)

DEBUG: Enabled data sources: ['Conserv', 'Coris', 'LI-COR']


Gathering LI-COR readings: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  2.31it/s]


Detailed information is saved in the log:

In [2]:
# Detailed info is saved in the log.
with open('../data/EnvironmentData.log', 'r') as file:
    for line in file.read().splitlines()[:10]:
        print(line)

2025-12-03 12:08:17,867 - EnvironmentData - INFO - Initialized Conserv client with 5 customers
2025-12-03 12:08:17,868 - EnvironmentData - INFO - Enabled data sources: ['Conserv', 'Coris', 'LI-COR']
2025-12-03 12:08:17,868 - EnvironmentData - INFO - Fetching Conserv historical data for all customers
2025-12-03 12:08:17,869 - EnvironmentData - INFO - Fetching Conserv data for period: 1764184097 to 1764788897
2025-12-03 12:08:17,869 - EnvironmentData - INFO - Running in test mode - only processing first customer: 333
2025-12-03 12:08:17,877 - EnvironmentData - INFO - Fetching data for customer 333
2025-12-03 12:08:17,877 - EnvironmentData - INFO - Exporting chunk for customer 333: 2025-11-26 19:08:17+00:00 to 2025-12-03 19:08:17+00:00
2025-12-03 12:08:17,878 - EnvironmentData - INFO - Starting export for customer 333: 2025-11-26 19:08:17+00:00 to 2025-12-03 19:08:17+00:00
2025-12-03 12:08:17,878 - EnvironmentData - INFO - Conserv API POST https://api.conserv.io/v1/sensors/export headers=

This saves our intermediate data to `data/sensor_readings.parquet`. 

Initially, we leave the data mostly as-is. We'll clean, add formatted dates, consolidate readings from the same device, etc. when moving to analytical steps, this preserves the source data so we can always change our mind later about how we decide to view it. 

However, at this point we are taking care to standardize the data format between different API sources. 

There are just a few columns because this is only historical data. We'll bring in current data shortly, and that will add more columns. 

In [3]:
import polars
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "Coris").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1764184097,1764789000,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.68,null,true
1764184997,1764789000,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.650002,null,true
1764185897,1764789000,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.739998,null,true
1764186797,1764789000,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",69.150002,null,true
1764187697,1764789000,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.830002,null,true


In [4]:
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "LI-COR").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1764184500,1764789014,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""","""RX Station 1_RH""","""RH""",null,48.868542,true
1764185400,1764789014,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""","""RX Station 1_RH""","""RH""",null,48.822765,true
1764186300,1764789014,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""","""RX Station 1_RH""","""RH""",null,48.604561,true
1764187200,1764789014,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""","""RX Station 1_RH""","""RH""",null,48.607613,true
1764188100,1764789014,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-2""","""RX Station 1_RH""","""RH""",null,48.705273,true


In [5]:
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "Conserv").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1764185537,1764788898,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.619995,null,true
1764186437,1764788898,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.494003,null,true
1764187337,1764788898,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.350006,null,true
1764188237,1764788898,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.169998,null,true
1764189137,1764788898,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""BFAST__BS___________ - Tempera…","""Temperature""",69.098007,null,true


# 2 Get Current Readings

Now we can start gathering and appending readings. There is a function `get_current_readings` that is run throughout the day, every 10 minutes for example. This function creates a parquet file at `data/new-readings` with the UTC as a filename. At the end of the day, all these readings will be consolidated into the database. 

Here is a sample of the readings:

In [6]:
# Wait 15 minutes to allow a new Conserv reading.
# import time
# time.sleep(15 * 60)  # Wait 15 minutes (900 seconds)

# envdt.get_current_readings()

# # Data is read into new-readings folder for consolidation at the end of the day.
# import os
# filename = os.listdir('../data/new-readings')[0]
# print(filename)
# polars.read_parquet('../data/new-readings/' + filename).sample(5)

# 3 Consolidate Readings

At the end of the day, new readings will be consolidated into the table. At the same time, the analytical tables will be generated. 

Analytical tables include:

* `device_readings.parquet`: Sensor readings reorganized to one row per Device and UTC, with measurements across columns vs measurements across rows.* 
* `sensors.parquet`: Information about the unique sensors. Includes information extracted from SensorName. Join this to Sensors during analysis to enhance with Building, Room, Direction, etc.
* `devices.parquet`: Information about unique devices. Includes information extracted from SensorName. 
* `utcs.parquet`: Information related to the UTC times in various datasets. Join to Sensors or Devices to enhance with Date, Time, Year, Hour, Weekday, etc.
* `sensor_readings_daily.parquet`: Example of sensor readings summarized to the daily level which reduces row count by 99.3% for even faster queries.
* `device_readings_daily.parquet`: Example of device readings summarized to the daily level which reduces row count by 99.3% for even faster queries. 

We fully re-generate analytical tables during each consolidation. The data is small enough that this is a fairly quick process, so re-running it in full each time will make it easy to ensure consistency as we expand and change the project. 

In [7]:
# To consolidate these into the database, run consolidate_readings.
envdt.consolidate_readings()

# New-readings files are gone now.
# They get deleted each day to confirm that they have been loaded into the database and prepare for the next consolidation.
if os.path.exists('../data/new-readings'):
    print(os.listdir('../data/new-readings'))

In [8]:
# Use this to re-run if you change the consolidation code.

# from EnvironmentData import EnvironmentData 
# envdt = EnvironmentData(
#     #days_back = 365 * 2,
#     days_back = 7,
#     coris_enabled = True,
#     licor_enabled = True,
#     conserv_enabled = True, 
#     #testing = True,
#     testing = False,
#     # Since we are running from the experiments/ folder, we need to tell the class to use the parent directory as home.
#     home_directory = ".."
# )
# envdt.close()
# del envdt

**^^ We want this to be empty** since we have consolidated new readings into the historical data. 

Once we are done working with data intake/processing, we close the class to release the file lock on the log file.

In [9]:
# When done, close the connection to the logs. 
envdt.close()

Let's look at the data we have now:

In [10]:
# Sensor Readings
# The first rows will be missing the extra fields like HexGatewayMac, etc.
#   I am pulling in some extra fields like DeviceID and DeviceName so we have that by historical. 
#   But some don't make sense to  backfill so they'll be null.
sensor_readings = polars.read_parquet('../data/sensor_readings.parquet')
sensor_readings.head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,SensorReadingUTC_SecondsFromPrior,Historical
i64,i32,str,str,str,str,str,str,f32,f32,i64,bool
1764184335,1764788898,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""BYCBA_0400410__N____ - RH""","""RH""",null,48.529999,null,true
1764185235,1764788898,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""BYCBA_0400410__N____ - RH""","""RH""",null,48.68,null,true
1764186135,1764788898,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""BYCBA_0400410__N____ - RH""","""RH""",null,48.529999,null,true
1764187035,1764788898,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""BYCBA_0400410__N____ - RH""","""RH""",null,48.599998,null,true
1764187935,1764788898,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""BYCBA_0400410__N____ - RH""","""RH""",null,48.25,null,true


In [11]:
# Recent rows will have the full data, aside from nulls due to a sensor not providing a reading type.
sensor_readings.tail()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,SensorReadingUTC_SecondsFromPrior,Historical
i64,i32,str,str,str,str,str,str,f32,f32,i64,bool
1764783900,1764789015,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""","""RX Station 1_Temperature""","""Temperature""",71.416176,null,null,true
1764784800,1764789015,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""","""RX Station 1_Temperature""","""Temperature""",71.300346,null,null,true
1764785700,1764789015,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""","""RX Station 1_Temperature""","""Temperature""",71.338959,null,null,true
1764786600,1764789015,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""","""RX Station 1_Temperature""","""Temperature""",71.454788,null,null,true
1764787500,1764789015,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""","""RX Station 1_Temperature""","""Temperature""",71.454788,null,null,true


In [12]:
# Device Readings.
device_readings = polars.read_parquet('../data/device_readings.parquet')
device_readings.head()

Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,SensorReadingUTC,QueryUTC,Historical,SensorReadingF,SensorReadingRh
str,str,str,str,str,str,i64,i32,bool,f32,f32
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:Temperatur…","""BYCBA_0400410__N____ - Tempera…","""Temperature, RH""",1764184335,1764788898,true,69.223999,48.529999
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:Temperatur…","""BYCBA_0400410__N____ - Tempera…","""Temperature, RH""",1764185235,1764788898,true,69.296005,48.68
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:Temperatur…","""BYCBA_0400410__N____ - Tempera…","""Temperature, RH""",1764186135,1764788898,true,69.115997,48.529999
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:Temperatur…","""BYCBA_0400410__N____ - Tempera…","""Temperature, RH""",1764187035,1764788898,true,69.242004,48.599998
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:Temperatur…","""BYCBA_0400410__N____ - Tempera…","""Temperature, RH""",1764187935,1764788898,true,69.350006,48.25


In [13]:
# Sensors
sensors = polars.read_parquet('../data/sensors.parquet')
sensors.head()

Source,SensorID,SensorName,SensorType,DeviceID,DeviceSerialFromName,BuildingID,Building,Room,CardinalDirection,DeviceName
str,str,str,str,str,str,str,str,str,str,str
"""Coris""","""coris:21378""","""RH KGL 21_D0B2""","""Humidity""","""coris:12167""","""D0B2""","""KGL""","""Kline Geology Laboratory""","""21""","""Not Indicated""","""Peabody TH-L Upper Great Hall …"
"""Coris""","""coris:21377""","""Temp KGL 21_D0B2""","""Temperature""","""coris:12167""","""D0B2""","""KGL""","""Kline Geology Laboratory""","""21""","""Not Indicated""","""Peabody TH-L Upper Great Hall …"
"""Conserv""","""conserv:333:c008914:RH""","""- RH""","""RH""","""conserv:333:c008914""",null,"""MALFORMED""","""Unknown""","""Unknown""",null,"""BYCBA_030030104_S___"""
"""Conserv""","""conserv:333:c009069:RH""","""- RH""","""RH""","""conserv:333:c009069""",null,"""MALFORMED""","""Unknown""","""Unknown""",null,"""BYCBA_B100B07_______"""
"""Conserv""","""conserv:333:c008949:RH""","""- RH""","""RH""","""conserv:333:c008949""",null,"""MALFORMED""","""Unknown""","""Unknown""",null,"""BYCBA_0200201S6_E___"""


In [14]:
# Devices. 
devices = polars.read_parquet('../data/devices.parquet')
devices.head()

Source,DeviceID,DeviceName,SensorIDs,SensorNames,SensorTypes,DeviceSerialFromName,BuildingID,Building,Room,CardinalDirection
str,str,str,str,str,str,str,str,str,str,str
"""Coris""","""coris:12167""","""Peabody TH-L Upper Great Hall …","""coris:21377, coris:21378""","""RH KGL 21_D0B2, Temp KGL 21_D0…","""Humidity, Temperature""","""D0B2""","""KGL""","""Kline Geology Laboratory""","""21""","""Not Indicated"""
"""Conserv""","""conserv:333:c009072""","""BBARCH0100001_______""","""conserv:333:c009072:RH, conser…","""- RH, - Temperature""","""RH, Temperature""",null,"""MALFORMED""","""Unknown""","""Unknown""",null
"""Conserv""","""conserv:333:c009073""","""BBARCHB100116_______""","""conserv:333:c009073:RH, conser…","""- RH, - Temperature""","""RH, Temperature""",null,"""MALFORMED""","""Unknown""","""Unknown""",null
"""Conserv""","""conserv:333:c009081""","""BCSC__01H103________""","""conserv:333:c009081:RH, conser…","""- RH, - Temperature""","""RH, Temperature""",null,"""MALFORMED""","""Unknown""","""Unknown""",null
"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:RH, conser…","""- RH, - Temperature""","""RH, Temperature""",null,"""MALFORMED""","""Unknown""","""Unknown""",null


In [15]:
# UTC Date/Time Info
utcs = polars.read_parquet('../data/utcs.parquet').head()
utcs.head()

UTC,datetime_utc,datetime_est,date,time,year,month,day_of_week,day_of_week_monday1_sunday7,hour_24,hour_12,am_pm
i64,datetime[μs],"datetime[μs, America/New_York]",date,time,i32,i8,str,i8,i8,i8,str
1764360204,2025-11-28 13:03:24,2025-11-28 08:03:24 EST,2025-11-28,08:03:24,2025,11,"""Friday""",5,8,8,"""AM"""
1764229133,2025-11-27 00:38:53,2025-11-26 19:38:53 EST,2025-11-26,19:38:53,2025,11,"""Wednesday""",3,19,7,"""PM"""
1764622350,2025-12-01 13:52:30,2025-12-01 08:52:30 EST,2025-12-01,08:52:30,2025,12,"""Monday""",1,8,8,"""AM"""
1764229143,2025-11-27 00:39:03,2025-11-26 19:39:03 EST,2025-11-26,19:39:03,2025,11,"""Wednesday""",3,19,7,"""PM"""
1764753439,2025-12-03 02:17:19,2025-12-02 21:17:19 EST,2025-12-02,21:17:19,2025,12,"""Tuesday""",2,21,9,"""PM"""


In [16]:
# Daily Sensor Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
sensor_readings_daily = polars.read_parquet('../data/sensor_readings_daily.parquet')
sensor_readings_daily.head()

Source,date,SensorID,row_count,SensorReadingF_sum,SensorReadingRh_sum,SensorReadingF_min,SensorReadingRh_min,SensorReadingF_max,SensorReadingRh_max
str,date,str,u32,f32,f32,f32,f32,f32,f32
"""Conserv""",2025-12-03,"""conserv:333:c008781:RH""",1,0.0,46.93,null,46.93,null,46.93
"""Conserv""",2025-12-03,"""conserv:333:c008781:Temperatur…",1,70.430008,0.0,70.430008,null,70.430008,null
"""Conserv""",2025-12-03,"""conserv:333:c008784:RH""",1,0.0,43.060001,null,43.060001,null,43.060001
"""Conserv""",2025-12-03,"""conserv:333:c008784:Temperatur…",1,70.753998,0.0,70.753998,null,70.753998,null
"""Conserv""",2025-12-03,"""conserv:333:c008924:RH""",1,0.0,45.959999,null,45.959999,null,45.959999


In [17]:
# Daily Device Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
device_readings_daily = polars.read_parquet('../data/device_readings_daily.parquet')
device_readings_daily.head()

Source,date,DeviceID,row_count,SensorReadingF_sum,SensorReadingRh_sum,SensorReadingF_min,SensorReadingRh_min,SensorReadingF_max,SensorReadingRh_max
str,date,str,u32,f32,f32,f32,f32,f32,f32
"""Conserv""",2025-12-03,"""conserv:333:c008781""",1,70.430008,46.93,70.430008,46.93,70.430008,46.93
"""Conserv""",2025-12-03,"""conserv:333:c008784""",1,70.753998,43.060001,70.753998,43.060001,70.753998,43.060001
"""Conserv""",2025-12-03,"""conserv:333:c008924""",1,69.764,45.959999,69.764,45.959999,69.764,45.959999
"""Conserv""",2025-12-03,"""conserv:333:c008949""",1,69.475998,43.27,69.475998,43.27,69.475998,43.27
"""Conserv""",2025-12-03,"""conserv:333:c009016""",1,72.014,40.650002,72.014,40.650002,72.014,40.650002


In [18]:
# Differentiate historical vs. cron readings by filtering on Historical = true.
import duckdb
duckdb.sql("""
    SELECT *
    FROM read_parquet('../data/device_readings.parquet') 
    WHERE Historical
    LIMIT 5
""").to_df()

,Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,SensorReadingUTC,QueryUTC,Historical,SensorReadingF,SensorReadingRh
0,Conserv,conserv:333:c008706,BYCBA_0400410__N____,"conserv:333:c008706:Temperature, conserv:333:c...","BYCBA_0400410__N____ - Temperature, BYCBA_0400...","Temperature, RH",1764184335,1764788898,True,69.223999,48.529999
1,Conserv,conserv:333:c008706,BYCBA_0400410__N____,"conserv:333:c008706:Temperature, conserv:333:c...","BYCBA_0400410__N____ - Temperature, BYCBA_0400...","Temperature, RH",1764185235,1764788898,True,69.296005,48.680000
2,Conserv,conserv:333:c008706,BYCBA_0400410__N____,"conserv:333:c008706:Temperature, conserv:333:c...","BYCBA_0400410__N____ - Temperature, BYCBA_0400...","Temperature, RH",1764186135,1764788898,True,69.115997,48.529999
3,Conserv,conserv:333:c008706,BYCBA_0400410__N____,"conserv:333:c008706:Temperature, conserv:333:c...","BYCBA_0400410__N____ - Temperature, BYCBA_0400...","Temperature, RH",1764187035,1764788898,True,69.242004,48.599998
4,Conserv,conserv:333:c008706,BYCBA_0400410__N____,"conserv:333:c008706:Temperature, conserv:333:c...","BYCBA_0400410__N____ - Temperature, BYCBA_0400...","Temperature, RH",1764187935,1764788898,True,69.350006,48.250000


In [19]:
duckdb.sql("""SELECT DISTINCT
    sr.Source,
    u.datetime_est
FROM '../data/sensor_readings.parquet' sr
LEFT JOIN '../data/utcs.parquet' u 
    ON sr.SensorReadingUTC = u.utc
WHERE sr.Source = 'LI-COR'
ORDER BY sr.SensorReadingUTC
""").to_df()

,Source,datetime_est
0,LI-COR,2025-11-26 05:15:00-07:00
1,LI-COR,2025-11-26 05:30:00-07:00
2,LI-COR,2025-11-26 05:45:00-07:00
3,LI-COR,2025-11-26 06:00:00-07:00
4,LI-COR,2025-11-26 06:15:00-07:00
...,...,...
666,LI-COR,2025-12-03 03:45:00-07:00
667,LI-COR,2025-12-03 04:00:00-07:00
668,LI-COR,2025-12-03 04:15:00-07:00
669,LI-COR,2025-12-03 04:30:00-07:00


# Validation & Alerts

There are two diagnostic files we can review to see if there are alerts or errors. 

In [20]:
# Read ../data/validation-results.csv
import pandas as pd
validation_results = pd.read_csv('../data/validation-results.csv')
validation_results

,run_datetime_est,run_utc,test_name,result,details
0,2025-12-03 14:10:16 EST,1764789016,required_columns_present,PASS,All 11 required columns are present in sensor ...
1,2025-12-03 14:10:16 EST,1764789016,column_data_types,PASS,All columns have the expected data types (e.g....
2,2025-12-03 14:10:16 EST,1764789016,non_null_values,PASS,Every sensor reading row has at least one non-...
3,2025-12-03 14:10:16 EST,1764789016,no_duplicate_readings,PASS,No duplicate readings found. Each sensor has u...
4,2025-12-03 14:10:16 EST,1764789016,sensor_name_consistency,PASS,All sensors have consistent names across all t...
5,2025-12-03 14:10:16 EST,1764789016,reading_interval_check,PASS,All consecutive readings are within 15 minutes...
6,2025-12-03 14:10:16 EST,1764789016,data_gaps_Conserv,WARN,Found 142 gaps in Conserv data where readings ...
7,2025-12-03 14:10:16 EST,1764789016,alerts_LI-COR,PASS,No alerts triggered for LI-COR. All sensor rea...
8,2025-12-03 14:10:16 EST,1764789016,alerts_Coris,PASS,No alerts triggered for Coris. All sensor read...
9,2025-12-03 14:10:16 EST,1764789016,alerts_Conserv,PASS,No alerts triggered for Conserv. All sensor re...


In [21]:
# Validation results that did not pass.
validation_results[validation_results['result'] != "PASS"]

,run_datetime_est,run_utc,test_name,result,details
6,2025-12-03 14:10:16 EST,1764789016,data_gaps_Conserv,WARN,Found 142 gaps in Conserv data where readings ...


In [22]:
# Read ../data/alerts.csv
alerts = pd.read_csv('../data/alerts.csv')
alerts.sample(15).sort_values(by='event_utc')

,event,Source,SensorID,SensorName,event_utc,event_datetime_est,event_end_utc,event_end_datetime_est,gap_minutes,reading_type,reading_value,threshold_min,threshold_max,detected_utc,detected_datetime_est
50,DATA_GAP,Conserv,conserv:333:c008924:Temperature,BYCBA_0300318_______ - Temperature,1764232478,2025-11-27 03:34:38 EST,1764234278,2025-11-27 04:04:38 EST,30.0,NaN,NaN,NaN,NaN,1764789016,2025-12-03 14:10:16 EST
90,DATA_GAP,Conserv,conserv:333:c009063:RH,BYCBA_B100B06_______ - RH,1764238874,2025-11-27 05:21:14 EST,1764240674,2025-11-27 05:51:14 EST,30.0,NaN,NaN,NaN,NaN,1764789016,2025-12-03 14:10:16 EST
47,DATA_GAP,Conserv,conserv:333:c008924:RH,BYCBA_0300318_______ - RH,1764311677,2025-11-28 01:34:37 EST,1764314377,2025-11-28 02:19:37 EST,45.0,NaN,NaN,NaN,NaN,1764789016,2025-12-03 14:10:16 EST
113,DATA_GAP,Conserv,conserv:333:c009072:RH,BBARCH0100001_______ - RH,1764327076,2025-11-28 05:51:16 EST,1764328876,2025-11-28 06:21:16 EST,30.0,NaN,NaN,NaN,NaN,1764789016,2025-12-03 14:10:16 EST
60,DATA_GAP,Conserv,conserv:333:c009016:RH,BYCBA_0200216_E_____ - RH,1764400051,2025-11-29 02:07:31 EST,1764401851,2025-11-29 02:37:31 EST,30.0,NaN,NaN,NaN,NaN,1764789016,2025-12-03 14:10:16 EST
74,DATA_GAP,Conserv,conserv:333:c009034:RH,BYCBA_0100112_N_____ - RH,1764437033,2025-11-29 12:23:53 EST,1764438833,2025-11-29 12:53:53 EST,30.0,NaN,NaN,NaN,NaN,1764789016,2025-12-03 14:10:16 EST
36,DATA_GAP,Conserv,conserv:333:c008903:RH,BYCBA_030030107_S___ - RH,1764447060,2025-11-29 15:11:00 EST,1764448860,2025-11-29 15:41:00 EST,30.0,NaN,NaN,NaN,NaN,1764789016,2025-12-03 14:10:16 EST
64,DATA_GAP,Conserv,conserv:333:c009023:RH,BYCBA_0200212_NE____ - RH,1764501617,2025-11-30 06:20:17 EST,1764503417,2025-11-30 06:50:17 EST,30.0,NaN,NaN,NaN,NaN,1764789016,2025-12-03 14:10:16 EST
121,DATA_GAP,Conserv,conserv:333:c009072:Temperature,BBARCH0100001_______ - Temperature,1764576376,2025-12-01 03:06:16 EST,1764578176,2025-12-01 03:36:16 EST,30.0,NaN,NaN,NaN,NaN,1764789016,2025-12-03 14:10:16 EST
94,DATA_GAP,Conserv,conserv:333:c009063:RH,BYCBA_B100B06_______ - RH,1764576378,2025-12-01 03:06:18 EST,1764578178,2025-12-01 03:36:18 EST,30.0,NaN,NaN,NaN,NaN,1764789016,2025-12-03 14:10:16 EST


In [23]:
# Group alerts by Source ans event.
alerts.groupby(['Source', 'event']).size().reset_index(name='count')

# No alerts (only data gaps) means all readings were within thresholds.

,Source,event,count
0,Conserv,DATA_GAP,142


Now you are ready to move onto analysis to get human-readable results (not indexed by UTC timestamps). See 2-examples-analysis.ipynb.